In [1]:
#pip install -e .

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats
import seaborn as sns
import pandas as pd
from datetime import datetime
import math
import re
from scipy.stats import norm

from noshow.preprocessing.load_data import (
    load_appointment_csv,
    process_postal_codes,
)

In [3]:
from pathlib import Path
data_path = Path().resolve().parents[0] / "data" / "raw"
appointments_df = load_appointment_csv(data_path / "poliafspraken_no_show.csv")

KeyboardInterrupt: 

In [ ]:
#filter for current location
appointments_df = appointments_df.loc[appointments_df["ziekenhuis"].isin(
            ["HagaZiekenhuis Den Haag"] #, "HagaZiekenhuis Zoetermeer"
        )]

In [ ]:
# --- Configuration ---
CLINIC_COL = 'hoofdagenda'
confidence = 0.95
Z = norm.ppf(1 - (1 - confidence) / 2)                      # z-score for 95%
TARGET_MOE_PP = 3.0            # desired margin of error in percentage points (±3%)

def n_for_moe_pp(moe_pp, z=Z):
    """
    Calculate minimum sample size n required for a given target margin of error (in % points)
    at 95% confidence for a proportion (worst-case p=0.5).
    """
    P = 0.5
    moe = moe_pp / 100.0  # convert to proportion (e.g., 3% -> 0.03)
    return math.ceil((z**2 * P * (1-P)) / (moe**2))


min_n = n_for_moe_pp(TARGET_MOE_PP)
print(f"Minimum n per clinic for ±{TARGET_MOE_PP:.0f}% precision at 95% confidence ≈ {min_n}")
clinic_counts = appointments_df[CLINIC_COL].value_counts(dropna=False)
print(f"Number of clinics in dataset: {len(clinic_counts)}")
clinic_counts.head()

In [ ]:
#show excluded clinics
clinics_keep = clinic_counts[clinic_counts >= min_n].index
clinics_excluded = clinic_counts[clinic_counts < min_n].index

print(f"Clinics kept: {len(clinics_keep)}")
print(f"Clinics excluded:")
print(clinic_counts[clinic_counts < min_n].sort_values())

In [ ]:
print(1718019-3693)
print(3693/1718019 * 100)

In [ ]:
total_value = clinic_counts[clinic_counts < min_n].sum()
print(total_value)

In [ ]:
#Filter based on threshold
appointments_df_filtered = appointments_df[appointments_df[CLINIC_COL].isin(clinics_keep)].copy()

print(f"Rows in original dataset: {len(appointments_df)}")
print(f"Rows after filtering:      {len(appointments_df_filtered)}")
print("Coverage: {:.2f}% of all records retained".format(
    100 * len(appointments_df_filtered) / len(appointments_df)
))

In [ ]:
appointments_df_filtered["no_show"] = "show"

# Setting 'no_show' to 'no_show' for rows with status "Is niet voldaan" and cancelationReason_code "N" or "NF"
appointments_df_filtered.loc[
    (appointments_df_filtered["status"] == "Is niet voldaan") & 
    (appointments_df_filtered["cancelationReason_code"].isin(["N", "NF"])),
    "no_show"
] = "no_show"

# Replacing 'no_show' with 1 and 'show' with 0
appointments_df_filtered["no_show"] = (
    appointments_df_filtered["no_show"].replace({"no_show": "1", "show": "0"}).astype(int)
)

# Grouping by 'hoofdagenda' (clinic) and calculating the no-show count, total count, and unique patients per clinic
no_show_rate = appointments_df_filtered.groupby('hoofdagenda').agg(
    no_show_count=('no_show', 'sum'),
    total_count=('no_show', 'size'),
    unique_patients=('pseudo_id', 'nunique')
).reset_index()

# Calculating the no-show rate for each clinic
no_show_rate['No_Show_Rate'] = no_show_rate['no_show_count'] / no_show_rate['total_count']

# Converting the no-show rate to percentage and formatting it to 3 decimal places
no_show_rate['No_Show_Rate'] = (no_show_rate['No_Show_Rate'] * 100).round(3).astype(str) + '%'

# Adding a column for total appointments per clinic
no_show_rate['Total_Appointments'] = no_show_rate['total_count']

# Displaying the result
display(no_show_rate[['hoofdagenda', 'No_Show_Rate', 'Total_Appointments', 'unique_patients']])


In [ ]:
appointments_df_filtered = appointments_df_filtered.loc[appointments_df_filtered["hoofdagenda"].isin(
    ["OOGHEELKUNDE"]
)]

In [ ]:
from noshow.features.feature_pipeline import create_features
from noshow.preprocessing.load_data import (
    load_appointment_csv,
    process_appointments,
    process_postal_codes,
)
from noshow.config import CLINIC_CONFIG

In [ ]:
all_postalcodes = process_postal_codes("../data/raw/NL.csv")
appointments_df = process_appointments(appointments_df_filtered, CLINIC_CONFIG)
appointments_features = create_features(appointments_df, all_postalcodes)

In [ ]:
# Resetting the index to move 'pseudo_id' from the index to a column
appointments_df_filtered_reset = appointments_df_filtered.reset_index()

# Checking unique values in the 'status' column to understand what '1' represents
print("Unique values in 'status':")
print(appointments_df_filtered_reset['status'].unique())

# Check how many NaN values are in 'cancelationReason_code'
print("Number of NaN values in 'cancelationReason_code':")
print(appointments_df_filtered_reset['cancelationReason_code'].isna().sum())

# Check unique values in 'cancelationReason_code' to see if they match expected values
print("Unique values in 'cancelationReason_code':")
print(appointments_df_filtered_reset['cancelationReason_code'].unique())

# If necessary, handle NaN values in 'cancelationReason_code' by replacing NaN with 'N'
appointments_df_filtered_reset['cancelationReason_code'].fillna('N', inplace=True)

# Check the updated 'cancelationReason_code' values after filling NaN
print("Updated 'cancelationReason_code' values after filling NaN:")
print(appointments_df_filtered_reset[['hoofdagenda', 'cancelationReason_code']].head())

# Set 'no_show' to 'show'
appointments_df_filtered_reset["no_show"] = "show"

# If status is 1 (assumed to be "Is niet voldaan") and cancelationReason_code is "N" or "NF", set no_show to 'no_show'
appointments_df_filtered_reset.loc[
    (appointments_df_filtered_reset["status"] == 1) & 
    (appointments_df_filtered_reset["cancelationReason_code"].isin(["N", "NF"])),
    "no_show"
] = "no_show"

# Replacing 'no_show' with 1 and 'show' with 0 and converting to integer type
appointments_df_filtered_reset["no_show"] = appointments_df_filtered_reset["no_show"].replace({"no_show": 1, "show": 0}).astype(int)

# Check the updated 'no_show' column
print("Updated no_show values after conversion:")
print(appointments_df_filtered_reset[['hoofdagenda', 'no_show']].head())

# Grouping by 'hoofdagenda' (clinic) and calculating the no-show count, total count, and unique patients per clinic
no_show_rate = appointments_df_filtered_reset.groupby('hoofdagenda').agg(
    no_show_count=('no_show', 'sum'),
    total_count=('no_show', 'size'),
    unique_patients=('pseudo_id', 'nunique')  # Now 'pseudo_id' is a regular column
).reset_index()

# Display no_show_count and total_count
print("Grouped by 'hoofdagenda':")
print(no_show_rate[['hoofdagenda', 'no_show_count', 'total_count']])

# Calculate the no-show rate for each clinic
no_show_rate['No_Show_Rate'] = no_show_rate['no_show_count'] / no_show_rate['total_count']

# Converting the no-show rate to percentage and formatting it to 3 decimal places
no_show_rate['No_Show_Rate'] = (no_show_rate['No_Show_Rate'] * 100).round(3).astype(str) + '%'

# Adding a column for total appointments per clinic
no_show_rate['Total_Appointments'] = no_show_rate['total_count']

# Displaying the result
display(no_show_rate[['hoofdagenda', 'No_Show_Rate', 'Total_Appointments', 'unique_patients']])


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

df = appointments_df_filtered.copy()

# If you have APPOINTMENT_DATE, we prefer age at appointment.
# Otherwise we fall back to "current year - BIRTH_YEAR".
def get_appointment_year(s):
    # Accept either datetime, string, or NaT
    if pd.api.types.is_datetime64_any_dtype(s):
        return s.dt.year
    try:
        return pd.to_datetime(s, errors='coerce').dt.year
    except Exception:
        return pd.Series([np.nan]*len(s), index=s.index)

if 'APPOINTMENT_DATE' in df.columns:
    appt_year = get_appointment_year(df['APPOINTMENT_DATE'])
else:
    appt_year = pd.Series([datetime.now().year]*len(df), index=df.index)

df['APPOINTMENT_YEAR'] = appt_year

# Normalize BIRTH_YEAR to numeric where possible (keep a copy for conformance check)
df['BIRTH_YEAR_RAW'] = df['BIRTH_YEAR']
df['BIRTH_YEAR'] = pd.to_numeric(df['BIRTH_YEAR'], errors='coerce')


In [ ]:
def run_clinic_quality_report(df, cfg: dict, skip_empty=False):
    """
    Generic, clinic-aware quality report for a single variable.
    """
    value_col   = cfg['value_col']
    clinic_col  = cfg['clinic_col']
    patient_col = cfg['patient_col']

    label = cfg.get('label', value_col)
    cast_fn = cfg.get('cast_fn', None)
    conformance_fn = cfg.get('conformance_fn', None)
    plausibility_fn = cfg.get('plausibility_fn', None)
    placeholder_set = cfg.get('placeholder_set', None)
    appointment_year_col = cfg.get('appointment_year_col', None)
    numeric_summary = bool(cfg.get('numeric_summary', False))

    if clinic_col not in df.columns:
        raise KeyError(f"Clinic column '{clinic_col}' not found.")
    if value_col not in df.columns:
        raise KeyError(f"Value column '{value_col}' not found.")
    
    # Skip patient-related logic if patient_col is None
    if patient_col not in df.columns:
        patient_col = None

    work = df.copy()

    # Keep raw column for reporting
    raw_col = f"{value_col}_RAW"
    if raw_col not in work.columns:
        work[raw_col] = work[value_col]

    # Optional casting (e.g., to numeric)
    cast_col = value_col
    if cast_fn is not None:
        cast_col = f"{value_col}_CAST"
        work[cast_col] = cast_fn(work[value_col])
    else:
        work[cast_col] = work[value_col]

    # Completeness
    work['DQ_missing'] = work[raw_col].isna()

    # If skip_empty is True, remove rows with empty values from the analysis
    if skip_empty:
        work = work.dropna(subset=[raw_col])

    # Conformance (True = FAIL so it’s easy to aggregate % fail)
    if conformance_fn is not None:
        ok_mask = conformance_fn(work)
        work['DQ_conformance_fail'] = ~ok_mask.fillna(False)
    else:
        work['DQ_conformance_fail'] = False

    # Placeholders
    if placeholder_set is not None:
        work['DQ_placeholder'] = work[raw_col].isin(placeholder_set)
    else:
        work['DQ_placeholder'] = False

    # Plausibility (True = IMPLAUSIBLE)
    if plausibility_fn is not None:
        work['DQ_implausible'] = plausibility_fn(work)
    else:
        work['DQ_implausible'] = False

    # Consistency within clinic: multiple values for the same patient
    if patient_col is not None:
        grp = work.groupby([clinic_col, patient_col], dropna=False)[cast_col].nunique()
        inconsistent_pairs = grp[grp > 1].reset_index()[[clinic_col, patient_col]]
        inconsistent_pairs['DQ_inconsistent'] = True
        work = work.merge(inconsistent_pairs, on=[clinic_col, patient_col], how='left')
        work['DQ_inconsistent'] = work['DQ_inconsistent'].fillna(False)
    else:
        work['DQ_inconsistent'] = False

    # Helpers
    def pct(series): 
        return 100.0 * series.mean() if series.size else np.nan

    # Per-clinic summary
    agg_dict = {
        'n_records':           (cast_col, 'size'),
        'missing_pct':         ('DQ_missing', pct),
        'implausible_pct':     ('DQ_implausible', pct),
        'conformance_fail_pct':('DQ_conformance_fail', pct),
        'placeholder_pct':     ('DQ_placeholder', pct),
    }
    if numeric_summary:
        agg_dict.update({
            f'min_{value_col}': (cast_col, 'min'),
            f'p50_{value_col}': (cast_col, 'median'),
            f'max_{value_col}': (cast_col, 'max'),
        })

    summary = (
        work.groupby(clinic_col, dropna=False)
            .agg(**agg_dict)
            .reset_index()
    )

    # Consistency rate
    if patient_col is not None:
        unique_patients = work.groupby(clinic_col)[patient_col].nunique().rename('unique_patients')
        inconsistent_patients = (
            work[[clinic_col, patient_col, 'DQ_inconsistent']]
            .drop_duplicates([clinic_col, patient_col])
            .groupby(clinic_col)['DQ_inconsistent']
            .sum()
            .rename('inconsistent_patients')
        )
        summary = summary.merge(unique_patients, on=clinic_col, how='left') \
                         .merge(inconsistent_patients, on=clinic_col, how='left')
        summary['unique_patients'] = summary['unique_patients'].fillna(0).astype(int, errors='ignore')
        summary['inconsistent_patients'] = summary['inconsistent_patients'].fillna(0).astype(int, errors='ignore')
        summary['inconsistent_rate_pct'] = np.where(
            summary['unique_patients'] > 0,
            100 * summary['inconsistent_patients'] / summary['unique_patients'],
            np.nan
        )

    # Details for drill-down
    details = {
        'implausible': work[work['DQ_implausible']].copy(),
        'conformance_fail': work[work['DQ_conformance_fail']].copy(),
        'placeholders': work[work['DQ_placeholder']].copy(),
        'inconsistent_detail': (
            work[work['DQ_inconsistent']]
              .groupby([clinic_col, patient_col], dropna=False)[cast_col]
              .agg(lambda s: sorted(pd.Series(s).dropna().unique().tolist()))
              .reset_index()
              .rename(columns={cast_col: f'{value_col}_values'})
            if patient_col is not None else pd.DataFrame()
        )
    }

    # Nice rounding for % columns
    for c in ['missing_pct','implausible_pct','conformance_fail_pct','placeholder_pct','inconsistent_rate_pct']:
        if c in summary.columns:
            summary[c] = summary[c].round(4)

    return summary, details


In [ ]:
def run_quality_section(df, cfg: dict, skip_empty=False):
    """
    Processes data for quality checks based on the configuration provided.
    Accepts the skip_empty flag to optionally remove missing values.
    """
    value_col = cfg['value_col']
    clinic_col = cfg['clinic_col']
    patient_col = cfg['patient_col']

    work = df.copy()

    # If skip_empty is True, remove rows with empty values
    if skip_empty:
        work = work.dropna(subset=[value_col])  # Drop rows with missing values in the column being checked

    # Run checks as usual (completeness, conformance, etc.)
    work['DQ_missing'] = work[value_col].isna()

    # Conformance (True = FAIL)
    if cfg.get('conformance_fn') is not None:
        conformance_fn = cfg['conformance_fn']
        work['DQ_conformance_fail'] = ~conformance_fn(work).fillna(False)
    else:
        work['DQ_conformance_fail'] = False

    # Drilldown for conformance failures
    drilldown_conformance_fail = work[work['DQ_conformance_fail']]

    # Plausibility check (True = IMPLAUSIBLE)
    if cfg.get('plausibility_fn') is not None:
        plausibility_fn = cfg['plausibility_fn']
        work['DQ_implausible'] = plausibility_fn(work)
    else:
        work['DQ_implausible'] = False

    # Drilldown for plausibility failures
    drilldown_implausible = work[work['DQ_implausible']]

    # Placeholders check (True = Placeholder)
    if cfg.get('placeholder_set') is not None:
        placeholder_set = cfg['placeholder_set']
        work['DQ_placeholder'] = work[value_col].isin(placeholder_set)
    else:
        work['DQ_placeholder'] = False

    # Drilldown for placeholder values
    drilldown_placeholders = work[work['DQ_placeholder']]

    # Completeness check (missing values)
    work['DQ_missing'] = work[value_col].isna()

    # Example of drilldown for missing values
    drilldown_missing = work[work['DQ_missing']]

    # Consistency within clinic (multiple values for the same patient)
    if patient_col is not None:
        grp = work.groupby([clinic_col, patient_col], dropna=False)[value_col].nunique()
        inconsistent_pairs = grp[grp > 1].reset_index()[[clinic_col, patient_col]]
        inconsistent_pairs['DQ_inconsistent'] = True
        work = work.merge(inconsistent_pairs, on=[clinic_col, patient_col], how='left')
        work['DQ_inconsistent'] = work['DQ_inconsistent'].fillna(False)
    else:
        work['DQ_inconsistent'] = False

    # Drilldown for inconsistencies
    drilldown_inconsistent = work[work['DQ_inconsistent']]

    # Summary generation with aggregation
    summary = work.groupby(clinic_col).agg({
        'DQ_missing': 'mean',  # percentage of missing values
        'DQ_conformance_fail': 'mean',  # percentage of conformance failures
        'DQ_implausible': 'mean',  # percentage of implausible values
        'DQ_placeholder': 'mean',  # percentage of placeholders
        'DQ_inconsistent': 'mean',  # percentage of inconsistencies (if patient_col is provided)
    })

    # Return the summary, work (for detailed checks), and all drilldowns
    return summary, work, {
        'conformance_fail': drilldown_conformance_fail,
        'implausible': drilldown_implausible,
        'placeholders': drilldown_placeholders,
        'missing': drilldown_missing,
        'inconsistent': drilldown_inconsistent
    }


In [ ]:
df = df.copy()
df['__clinic_norm'] = df['hoofdagenda'].astype(str).str.strip()

In [ ]:
# --- Uniqueness: semantic duplicates per clinic (abs + %) ---

# Columns that define a "same appointment" (adjust to your schema)
cols_to_check = ['pseudo_id', 'hoofdagenda', 'start', 'end', 'status', 'soort_cons', 'created', 'arrival']
cols_to_check = [c for c in cols_to_check if c in df.columns]  # keep only those that exist

# Flag rows that have an identical twin across the defining columns
sem_dupes_flag = df.duplicated(subset=cols_to_check, keep=False)

# Count semantic duplicates per clinic (absolute)
sem_dupe_counts = (
    df[sem_dupes_flag]
      .groupby('hoofdagenda', dropna=False)
      .size()
      .reset_index(name='semantic_duplicate_count')
)

# Total appointments per clinic
total_per_clinic = (
    df.groupby('hoofdagenda', dropna=False)
      .size()
      .reset_index(name='total_appointments')
)

# Merge and compute percentage
sem_dupe_summary = (
    total_per_clinic
      .merge(sem_dupe_counts, on='hoofdagenda', how='left')
      .fillna({'semantic_duplicate_count': 0})
)

sem_dupe_summary['semantic_duplicate_count'] = sem_dupe_summary['semantic_duplicate_count'].astype(int)
sem_dupe_summary['semantic_duplicate_pct'] = (
    100.0 * sem_dupe_summary['semantic_duplicate_count'] / sem_dupe_summary['total_appointments']
)

sem_dupe_summary = sem_dupe_summary.sort_values('semantic_duplicate_pct', ascending=False)
sem_dupe_summary['semantic_duplicate_pct'] = sem_dupe_summary['semantic_duplicate_pct'].round(4)

sem_dupe_summary = sem_dupe_summary.sort_values('hoofdagenda', ascending=True)

sem_dupe_summary

In [ ]:
# --- APP_ID ---
app_id_cfg = {
    'label': 'APP_ID',
    'value_col': 'APP_ID',
    'clinic_col': '__clinic_norm',
    'patient_col': None,
    #'cast_fn': lambda s: pd.to_numeric(s, errors='coerce'),
    'conformance_fn': lambda d: d['APP_ID'].apply(
        lambda x: (
            pd.notna(x)
            and (
                # Case 1: it's an integer or int-like float
                (isinstance(x, (int, float)) and not pd.isna(x) and 1e7 <= x < 1e10)
                # Case 2: it's a string of 8–9 digits
                or (isinstance(x, str) and bool(re.fullmatch(r'\d{8,9}', x.strip())))
            )
        )
    ),
    # no plausibility, no placeholders, no numeric summary
}

summary_by_clinic, details_app_id, extras_app_id = run_quality_section(
    df,
    app_id_cfg,
)

In [ ]:
# --- pseudo_id ---
clinic_col = '__clinic_norm'   # or 'hoofdagenda'
id_col     = 'pseudo_id'

# Start from your details_pseudo (row-level flags), or rebuild flags quickly:
d = df[[clinic_col, id_col]].copy()
d[id_col] = d[id_col].astype('string')

flags = pd.DataFrame({
    clinic_col: d[clinic_col],
    id_col: d[id_col],
})
flags['ID_missing'] = d[id_col].isna() | (d[id_col].str.len() == 0)
flags['ID_conformance_fail'] = ~d[id_col].str.fullmatch(r'[0-9A-Fa-f]{64}', na=False)
flags['ID_placeholder'] = d[id_col].isin({ '0'*64, 'F'*64, 'f'*64 })

# Ensure pure booleans (no NA) so .sum() returns integer counts
for c in ['ID_missing','ID_conformance_fail','ID_placeholder']:
    flags[c] = flags[c].fillna(False).astype(bool)

# Denominator per clinic
n = flags.groupby(clinic_col).size().to_frame('n_records')

# Numerators per clinic (counts of True)
num = (
    flags.groupby(clinic_col)[['ID_missing','ID_conformance_fail','ID_placeholder']]
         .sum()
         .astype('int64')
)

# Percentages (force float)
pct = (num.div(n['n_records'], axis=0) * 100.0).astype('float64')
pct.columns = [c + '_pct' for c in pct.columns]

# Unique IDs per clinic (descriptive)
uniq = flags.groupby(clinic_col)[id_col].nunique().to_frame('unique_ids')

# Assemble summary
summary_pseudo = (
    n.join(pct).join(uniq)
      .reset_index()
)

# Nice rounding for display
for c in ['ID_missing_pct','ID_conformance_fail_pct','ID_placeholder_pct']:
    summary_pseudo[c] = summary_pseudo[c].round(4)

summary_pseudo


In [ ]:
# --- HOOFDAGENDA  ---
hoofdagenda_cfg = {
    'label': 'hoofdagenda',
    'value_col': 'hoofdagenda',
    'clinic_col': '__clinic_norm',
    'patient_col': 'pseudo_id',

    'conformance_fn': lambda d: d['hoofdagenda'].apply(
        lambda x: (
            isinstance(x, str)
            and bool(re.fullmatch(r"[A-Za-zÀ-ÖØ-öø-ÿ0-9&/() .,'\"-]{2,60}", x.strip()))
        )
    ),
    'plausibility_fn': lambda d: pd.Series(False, index=d.index, dtype=bool),
    'numeric_summary': False,
}


summary_by_clinic, details_hoofdagenda = run_clinic_quality_report(
    df,
    hoofdagenda_cfg,
)

display(summary_by_clinic)


In [ ]:
# --- HOOFDAGENDA_ID ---
hoofdagenda_id_cfg = {
    'label': 'hoofdagenda_id',
    'value_col': 'hoofdagenda_id',
    'clinic_col': '__clinic_norm',
    'patient_col': 'pseudo_id',

    'conformance_fn': lambda d: d['hoofdagenda_id'].apply(
        lambda x: (
            isinstance(x, str)
            and bool(re.fullmatch(r'[A-Z0-9]{6}', x.strip().upper()))
        )
    ),
    'plausibility_fn': lambda d: pd.Series(False, index=d.index, dtype=bool),
    'numeric_summary': False,
    
}

summary_by_clinic, details_hoofdagenda_id, = run_clinic_quality_report(
    df,
    hoofdagenda_id_cfg,
)
display(summary_by_clinic)

In [ ]:
# --- SUBAGENDA_ID ---
subagenda_id_cfg = {
    'label': 'subagenda_id',
    'value_col': 'subagenda_id',
    'clinic_col': '__clinic_norm',
    'patient_col': None,
    'conformance_fn': lambda d: d['subagenda_id'].apply(
        lambda x: (
            isinstance(x, str)
            and bool(re.fullmatch(r'[A-Z0-9]{6}', x.strip().upper()))
        )
    ),
    'plausibility_fn': lambda d: pd.Series(False, index=d.index, dtype=bool),
    'numeric_summary': False,
}

summary_by_clinic, details_subagenda_id = run_clinic_quality_report(
    df,
    subagenda_id_cfg,
)
display(summary_by_clinic)

In [ ]:
# --- SPECIALTY_CODE ---
specialty_code_cfg = {
    'label': 'specialty_code',
    'value_col': 'specialty_code',
    'clinic_col': '__clinic_norm',     # normalized clinic column
    'patient_col': None,

    'conformance_fn': lambda d: d['specialty_code'].apply(
        lambda x: isinstance(x, str) and bool(re.fullmatch(r'[A-Za-z]{3}', x.strip()))
    ),

    # Plausibility: not applicable for categorical codes
    'plausibility_fn': lambda d: pd.Series(False, index=d.index, dtype=bool),
    'numeric_summary': False,
}

summary_by_clinic, details_specialty_code = run_clinic_quality_report(
    df,
    specialty_code_cfg,
)
display(summary_by_clinic)

In [ ]:
# --- SOORT_CONSULT ---
soort_consult_cfg = {
    'label': 'soort_consult',
    'value_col': 'soort_consult',
    'clinic_col': '__clinic_norm',     # normalized clinic column
    'patient_col': None,

    'conformance_fn': lambda d: d['hoofdagenda_id'].apply(
        lambda x: (
            isinstance(x, str)
            and bool(re.fullmatch(r'[A-Z0-9]{6}', x.strip().upper()))
        )
    ),
    # Plausibility: not applicable for categorical codes
    'plausibility_fn': lambda d: pd.Series(False, index=d.index, dtype=bool),
    'numeric_summary': False,
}

summary_by_clinic, details_soort_consult, extras_soort_consult = run_clinic_quality_report(
    df,
    soort_consult_cfg,
)
display(summary_by_clinic)


In [ ]:
# --- afspraak_code ---
afspraak_code_cfg = {
    'label': 'afspraak_code',
    'value_col': 'afspraak_code',
    'clinic_col': '__clinic_norm',
    'patient_col': None,

    'conformance_fn': lambda d: d['afspraak_code'].apply(
        lambda x: (
            isinstance(x, str)
            and bool(re.fullmatch(r'[A-Za-z0-9&/() .,\-\'"+*]*', x.strip()))  # Allow alphanumeric and some special chars
        ) if pd.notna(x) else False  # Fail if missing (NaN)
    ),
    'plausibility_fn': lambda d: pd.Series(False, index=d.index, dtype=bool),
    'numeric_summary': False,
}

summary_by_clinic, details_afspraak_code = run_clinic_quality_report(
    df,
    afspraak_code_cfg,
)
display(summary_by_clinic)


In [ ]:
# --- START  ---
start_cfg = {
    'label': 'start',
    'value_col': 'start',
    'clinic_col': 'hoofdagenda',
    'patient_col': None,
    'conformance_fn': lambda d: (
        pd.to_datetime(d['start'], errors='coerce', format='%Y-%m-%d %H:%M:%S').notna()
        | d['start'].apply(lambda x: isinstance(x, (pd.Timestamp, datetime)))
    ),

    'plausibility_fn': lambda d: (
        (pd.to_datetime(d['start'], errors='coerce', format='%Y-%m-%d %H:%M:%S') < pd.Timestamp('2018-01-01')) |
        (pd.to_datetime(d['start'], errors='coerce', format='%Y-%m-%d %H:%M:%S') > pd.Timestamp('2026-12-31'))
    ),

    'placeholder_set': {
        '1900-01-01 00:00:00', '2100-01-01 00:00:00',
        pd.Timestamp('1900-01-01 00:00:00'), pd.Timestamp('2100-01-01 00:00:00')
    },

    'numeric_summary': False,  # dates aren't summarized numerically here
}

summary_start, details_start, extras_start = run_clinic_quality_report(df, start_cfg)

In [ ]:
# --- END  ---
end_cfg = {
    'label': 'end',
    'value_col': 'end',
    'clinic_col': 'hoofdagenda',
    'patient_col': None,
    'conformance_fn': lambda d: (
        pd.to_datetime(d['end'], errors='coerce', format='%Y-%m-%d %H:%M:%S').notna()
        | d['end'].apply(lambda x: isinstance(x, (pd.Timestamp, datetime)))
    ),

    'plausibility_fn': lambda d: (
        (pd.to_datetime(d['end'], errors='coerce', format='%Y-%m-%d %H:%M:%S') < pd.Timestamp('2018-01-01')) |
        (pd.to_datetime(d['end'], errors='coerce', format='%Y-%m-%d %H:%M:%S') > pd.Timestamp('2026-12-31'))
    ),

    'placeholder_set': {
        '1900-01-01 00:00:00', '2100-01-01 00:00:00',
        pd.Timestamp('1900-01-01 00:00:00'), pd.Timestamp('2100-01-01 00:00:00')
    },

    'numeric_summary': False,  # dates aren't summarized numerically here
}

summary_end, details_end, extras_end = run_clinic_quality_report(df, end_cfg)
display(summary_end)

In [ ]:
# --- ARRIVAL  ---
arrival_cfg = {
    'label': 'arrival',
    'value_col': 'arrival',
    'clinic_col': 'hoofdagarrivala',
    'patient_col': None,
    'conformance_fn': lambda d: (
        pd.to_datetime(d['arrival'], errors='coerce', format='%Y-%m-%d %H:%M:%S').notna()
        | d['arrival'].apply(lambda x: isinstance(x, (pd.Timestamp, datetime)))
    ),

    'plausibility_fn': lambda d: (
        (pd.to_datetime(d['arrival'], errors='coerce', format='%Y-%m-%d %H:%M:%S') < pd.Timestamp('2018-01-01')) |
        (pd.to_datetime(d['arrival'], errors='coerce', format='%Y-%m-%d %H:%M:%S') > pd.Timestamp('2026-12-31'))
    ),

    'placeholder_set': {
        '1900-01-01 00:00:00', '2100-01-01 00:00:00',
        pd.Timestamp('1900-01-01 00:00:00'), pd.Timestamp('2100-01-01 00:00:00')
    },

    'numeric_summary': False,  # dates aren't summarized numerically here
}

summary_arrival, details_arrival = run_clinic_quality_report(df, end_cfg)
display(summary_arrival)

In [ ]:
# --- created  ---
created_cfg = {
    'label': 'created',
    'value_col': 'created',
    'clinic_col': 'hoofdagenda',
    'patient_col': None,
    'conformance_fn': lambda d: (
        pd.to_datetime(d['created'], errors='coerce', format='%Y-%m-%d %H:%M:%S').notna()
        | d['created'].apply(lambda x: isinstance(x, (pd.Timestamp, datetime)))
    ),

    'plausibility_fn': lambda d: (
        (pd.to_datetime(d['created'], errors='coerce', format='%Y-%m-%d %H:%M:%S') < pd.Timestamp('2018-01-01')) |
        (pd.to_datetime(d['created'], errors='coerce', format='%Y-%m-%d %H:%M:%S') > pd.Timestamp('2026-12-31'))
    ),

    'placeholder_set': {
        '1900-01-01 00:00:00', '2100-01-01 00:00:00',
        pd.Timestamp('1900-01-01 00:00:00'), pd.Timestamp('2100-01-01 00:00:00')
    },

    'numeric_summary': False,  # dates aren't summarized numerically here
}

summary_created, details_created = run_clinic_quality_report(df, end_cfg)
display(summary_created)

In [ ]:
# --- created  ---
created_cfg = {
    'label': 'created',
    'value_col': 'created',
    'clinic_col': 'hoofdagcreateda',
    'patient_col': None,
    'conformance_fn': lambda d: (
        pd.to_datetime(d['created'], errors='coerce', format='%Y-%m-%d %H:%M:%S').notna()
        | d['created'].apply(lambda x: isinstance(x, (pd.Timestamp, datetime)))
    ),

    'plausibility_fn': lambda d: (
        (pd.to_datetime(d['created'], errors='coerce', format='%Y-%m-%d %H:%M:%S') < pd.Timestamp('2018-01-01')) |
        (pd.to_datetime(d['created'], errors='coerce', format='%Y-%m-%d %H:%M:%S') > pd.Timestamp('2026-12-31'))
    ),

    'placeholder_set': {
        '1900-01-01 00:00:00', '2100-01-01 00:00:00',
        pd.Timestamp('1900-01-01 00:00:00'), pd.Timestamp('2100-01-01 00:00:00')
    },
    'numeric_summary': False, 
}

summary_created, details_created = run_clinic_quality_report(df, end_cfg)
display(summary_created)

In [ ]:
# --- minutesDuration ---
duration_cfg = {
    'label': 'minutesDuration',
    'value_col': 'minutesDuration',
    'clinic_col': '__clinic_norm',
    'patient_col': None,

    'conformance_fn': lambda d: d['minutesDuration'].apply(
        lambda x: pd.to_numeric(x, errors='coerce') if pd.notna(x) else np.nan
    ).notna(),  # Check if it's not NaN (valid number)

    'plausibility_fn': lambda d: (
        pd.to_numeric(d['minutesDuration'], errors='coerce') < 0) | 
        (pd.to_numeric(d['minutesDuration'], errors='coerce') > 1440),
    'placeholder_set': {0},
    'numeric_summary': False,
}

summary_dur, details_dur = run_clinic_quality_report(df, duration_cfg)
display(summary_dur)


In [ ]:
# --- status ---
status_cfg = {
    'label': 'status',
    'value_col': 'status',
    'clinic_col': '__clinic_norm',
    'patient_col': None,
    'conformance_fn': lambda d: d['status'].apply(
        lambda x: x in ['Is voldaan', 'Is niet voldaan', 'Onbekend']
    ),
    'plausibility_fn': lambda d: pd.Series(False, index=d.index, dtype=bool),
    'placeholder_set': {'Onbekend'},
    'numeric_summary': False,  # No numeric summary for status
}

summary_status, details_status = run_clinic_quality_report(df, status_cfg)
display(summary_status)

In [ ]:
# --- status_code_original  ---
status_code_original_cfg = {
    'label': 'status_code_original',
    'value_col': 'status_code_original',
    'clinic_col': '__clinic_norm',
    'patient_col': None,   
    'conformance_fn': lambda d: d['status_code_original'] == d['status'], 
    'plausibility_fn': lambda d: pd.Series(False, index=d.index, dtype=bool),  # No plausibility check needed
    'placeholder_set': {'Onbekend'},
    'numeric_summary': False, 

summary_status_code_original, details_status_code_original = run_clinic_quality_report(df, status_code_original_cfg)
display(summary_status_code_original)


In [ ]:
# --- cancelationReason_code  ---
cancelationReason_code_cfg = {
    'label': 'cancelationReason_code',
    'value_col': 'cancelationReason_code',
    'clinic_col': '__clinic_norm', 
    'patient_col': None, 
    'conformance_fn': lambda d: d['cancelationReason_code'].apply(
        lambda x: x in ['N', 'Q', 'NF'] or pd.isna(x)
    ),
    'plausibility_fn': lambda d: pd.Series(False, index=d.index, dtype=bool), 
    'placeholder_set': {'Onbekend'},
    'numeric_summary': False,
}

summary_cancelationReason_code, details_cancelationReason_code = run_clinic_quality_report(df, cancelationReason_code_cfg)

display(summary_cancelationReason_code)


In [ ]:
def conformance_check(row):
    """
    Function to check the conformance of cancelationReason_display against cancelationReason_code.
    This checks if cancelationReason_display corresponds to the correct cancelationReason_code.
    """
    # If both 'cancelationReason_display' and 'cancelationReason_code' are empty (NaN), treat as valid
    if pd.isna(row['cancelationReason_display']) and pd.isna(row['cancelationReason_code']):
        return True
    
    # If 'cancelationReason_display' is not NaN, perform conformance check
    if pd.notna(row['cancelationReason_display']):
        # Check specific conditions for each cancelationReason_display and corresponding cancelationReason_code
        if row['cancelationReason_display'] == 'Patient niet verschenen (of te laat gemeld)' and row['cancelationReason_code'] == 'N':
            return True
        elif row['cancelationReason_display'] == 'Patient niet bereikbaar (telefonisch consult)' and row['cancelationReason_code'] == 'Q':
            return True
        elif row['cancelationReason_display'] == 'No show (geen factuur)' and row['cancelationReason_code'] == 'NF':
            return True
    
    # If any other case fails, return False
    return False


def plausibility_check(d):
    """
    This function doesn't perform any plausibility check for this field.
    """
    return pd.Series(False, index=d.index, dtype=bool)

# --- cancelationReason_display configuration ---
cancelationReason_display_cfg = {
    'label': 'cancelationReason_display',
    'value_col': 'cancelationReason_display',
    'clinic_col': '__clinic_norm', 
    'patient_col': None,
    # Use the regular function for conformance
    'conformance_fn': lambda d: d.apply(conformance_check, axis=1),  # Apply conformance check row-wise

    'plausibility_fn': plausibility_check,
    'placeholder_set': {'Onbekend'},
    'numeric_summary': False,  # No numeric summary for cancelationReason_display
}

summary_cancelationReason_display, details_cancelationReason_display = run_clinic_quality_report(df, cancelationReason_display_cfg)
display(summary_cancelationReason_display)

In [ ]:
from datetime import datetime

# Get the current year
current_year = datetime.now().year

birthyear_cfg = {
    'label': 'BIRTH_YEAR',
    'value_col': 'BIRTH_YEAR', 
    'clinic_col': '__clinic_norm', 
    'patient_col': 'pseudo_id',
    'appointment_year_col': 'APPOINTMENT_YEAR',

    'conformance_fn': lambda d: (
        d['BIRTH_YEAR'].astype(str).str.fullmatch(r'\d{4}', na=False)  # Check if it's a valid 4-digit year
    ),
    # Plausibility check: Calculate age and ensure it's between 0 and 120 years
    'plausibility_fn': lambda d: (
        (d['BIRTH_YEAR'] < 1905) | (d['BIRTH_YEAR'] > current_year)
    ),
    'placeholder_set': {0},
    'numeric_summary': False, 
}

summary_by_clinic, details_birth = run_clinic_quality_report(df, birthyear_cfg)

display(summary_by_clinic)


In [ ]:
print(df.dtypes)

In [ ]:
all_postalcodes = process_postal_codes("../data/raw/NL.csv")
print(all_postalcodes)

In [ ]:
from typing import Union

all_postalcodes = process_postal_codes("../data/raw/NL.csv")

# --- address_postalCodeNumbersNL ---
address_postalCode_cfg = {
    'label': 'address_postalCodeNumbersNL',
    'value_col': 'address_postalCodeNumbersNL',
    'clinic_col': '__clinic_norm', 
    'patient_col': 'pseudo_id',

    'conformance_fn': lambda d: (
        d['address_postalCodeNumbersNL'].astype(str).str.match(r'^\d{4}$')  # Check for 4-digit numeric values
    ),

    'plausibility_fn': lambda d: (
        ~d['address_postalCodeNumbersNL'].astype(str).isin(all_postalcodes.index)  # Postal code is not in the valid list
    ),

    
    'placeholder_set': {'0000'},
    'numeric_summary': False, 
}

summary_postal_code, details_postal_code = run_clinic_quality_report(df, address_postalCode_cfg)
display(summary_postal_code)


In [ ]:
# --- soort_cons ---
soort_cons_cfg = {
    'label': 'soort_cons',
    'value_col': 'soort_cons',
    'clinic_col': '__clinic_norm', 
    'patient_col': None,
    'conformance_fn': lambda d: d['soort_cons'].isin(['Fysiek', 'Telefonisch', 'Schriftelijk', 'Video', 'Huisbezoek', 'Onbekend']),    
    # No plausibility check needed as this is categorical
    'plausibility_fn': lambda d: pd.Series(False, index=d.index, dtype=bool),
    'placeholder_set': {'Onbekend'},
    
    'numeric_summary': False,
}

summary_soort_cons, details_soort_cons = run_clinic_quality_report(df, soort_cons_cfg)
display(summary_soort_cons)

In [ ]:
# --- ConsultTypeDescription ---
consult_type_cfg = {
    'label': 'ConsultTypeDescription',
    'value_col': 'ConsultTypeDescription',
    'clinic_col': '__clinic_norm',
    'patient_col': None,
    # Conformance check: Ensure it matches one of the allowed values
    'conformance_fn': lambda d: d['ConsultTypeDescription'].isin([
        'Herhaling', 
        'Eerste', 
        'Intercollegiaal', 
        'Medebehandeling', 
        'Screen to screen', 
        'Traumatologisch', 
        'Keuring', 
        'Samen beslissen',
        'Geen'
    ]),
    # No plausibility check needed as this is categorical
    'plausibility_fn': lambda d: pd.Series(False, index=d.index, dtype=bool),
    'placeholder_set': {'Geen'},
    'numeric_summary': False, 
}

summary_consult_type, details_consult_type = run_clinic_quality_report(df, consult_type_cfg)
display(summary_consult_type)

In [ ]:
# --- gender ---
gender_cfg = {
    'label': 'gender',
    'value_col': 'gender',
    'clinic_col': '__clinic_norm', 
    'patient_col': 'pseudo_id',
    'conformance_fn': lambda d: d['gender'].isin(['Man', 'Vrouw', 'Onbekend']),
    # No plausibility check needed as this is categorical
    'plausibility_fn': lambda d: pd.Series(False, index=d.index, dtype=bool),
    'placeholder_set': {'Onbekend'},
    'numeric_summary': False,
}


summary_gender, details_gender = run_clinic_quality_report(df, gender_cfg)
display(summary_gender)

In [ ]:
appointments_df[appointments_df['minutesDuration'] == 1440]

In [ ]:
# Check the few highest 'minutesDuration' values
top_minutes_duration = df['minutesDuration'].apply(pd.to_numeric, errors='coerce').nlargest(10)  # Top 10 highest
print(top_minutes_duration)

In [ ]:
appointments_df['address_postalCodeNumbersNL'].value_counts()

In [ ]:
appointments_df['gender'].value_counts()

In [ ]:
empty_start_rows = appointments_df[
    appointments_df['start'].isna() | (appointments_df['start'].astype(str).str.strip() == '')
]
print(empty_start_rows)

In [ ]:
appointments_df['ConsultTypeDescription'].value_counts()